# 03 — Cottonwood training data from NAIP

**Phase 2 of the [cottonwood research plan](../../planning/Cheyenne_River_cottonwood_research_plan.md).**
The goal of this notebook is *training labels*, produced automatically, so that nobody has to
hand-digitize cottonwood — labels drawn by hand on this river would not transfer to the next one.

The method is a label factory in four steps, all inside the valley bottom from the VBET notebooks:

| Step | What it does |
|---|---|
| **Segment** | cut the NAIP image into crown-sized superpixels (SLIC) |
| **Describe** | give each segment colour, greenness, roughness, and shadow features |
| **Cluster** | k-means finds the natural groups, with no classes told to it in advance |
| **Assign** | a short, physical rule set names each *cluster* — not each pixel |

Naming clusters rather than pixels is what keeps this honest and portable: there are ten decisions
to audit, and you can read every one of them in a table.

**In**: a VBET run (`01s` or `01`) + NAIP from the Planetary Computer.
**Out**: a class raster, gallery patch polygons, a stratified validation sample for the group to
interpret, and a run manifest.

> **A note on what we can and cannot call this.** Four-band leaf-on NAIP separates *woody riparian*
> from grass and bare ground well. It does **not** reliably separate cottonwood from willow or shrub
> thicket (§5.1 of the plan). The mapped class is therefore named **`riparian_woody`**, not
> "cottonwood", until a canopy height model says otherwise.

## 0. Setup

In [ ]:
import os, sys

# PROJ/GDAL paths must be set before any geospatial import: the Jupyter kernel starts
# without `conda activate`, so PROJ cannot otherwise find its database.
def _find_share(name):
    for base in (sys.prefix, sys.base_prefix):
        p = os.path.join(base, "share", name)
        if os.path.isdir(p):
            return p
    return None

_proj, _gdal = _find_share("proj"), _find_share("gdal")
if _proj:
    os.environ["PROJ_DATA"] = os.environ["PROJ_LIB"] = _proj
if _gdal:
    os.environ.setdefault("GDAL_DATA", _gdal)

import json, time
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap

import rasterio
import rasterio.plot
from rasterio.merge import merge
from rasterio.warp import reproject, Resampling
from rasterio.features import rasterize, shapes
from shapely.geometry import box, shape

from scipy.ndimage import (uniform_filter, maximum_filter, distance_transform_edt,
                           binary_closing, label as cc_label)
from skimage.segmentation import slic, mark_boundaries
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

import pystac_client, planetary_computer


def _repo_root():
    """Walk up from the working directory to the repo root."""
    here = Path.cwd().resolve()
    for p in (here, *here.parents):
        if (p / ".git").exists() or (p / "environment.yml").is_file():
            return p
    raise RuntimeError("Could not find the repo root from " + str(here))

REPO     = _repo_root()
DATA_DIR = REPO / "data"
RUNS_DIR = REPO / "runs"          # manifests live here because data/ is gitignored
RUNS_DIR.mkdir(parents=True, exist_ok=True)

# ---- Figures ----
# figures/ is gitignored, but tracked via .gitkeep so it exists on a fresh clone —
# nothing to set up after cloning. Contents stay out of git because the repo is
# public and an executed run embeds a 60 cm gallery map (research plan §11).
FIG_DIR = REPO / "figures"
FIG_DPI = 300

def savefig(name, dpi=FIG_DPI):
    """Save the current figure to figures/<FIG_SUBDIR>/<name>.png.

    Call this BEFORE plt.show(): showing a figure can clear it, and you would
    silently save a blank page. FIG_SUBDIR is set in the config cell.
    """
    out = FIG_DIR / FIG_SUBDIR
    out.mkdir(parents=True, exist_ok=True)
    path = out / f"{name}.png"
    plt.savefig(path, dpi=dpi, bbox_inches="tight")
    print(f"figure -> {path.relative_to(REPO)}")
    return path

print("Imports OK")
print(f"  repo : {REPO}")

## 1. Configuration

Everything you might change lives in this one cell. Pointing this notebook at another site, another
year, or the full 13TFJ tile is an edit here and nowhere else.

In [ ]:
# ---- Which VBET run supplies the valley bottom ----
# "smoketest_redshirt" is the 20 x 20 km box from 01s and is what exists today.
# When Phase 1b finishes set this to "vbet_13TFJ"; nothing else in this notebook changes.
VBET_RUN   = "smoketest_redshirt"
VBET_DIR   = DATA_DIR / VBET_RUN
VBET_RES_M = 30                      # resolution the VBET rasters were built at

# ---- The analysis window ----
# NAIP at 60 cm is ~2,800 pixels per km per band, so we work a window at a time.
# The walkthrough reaches are gauge-anchored: one window per 8-digit USGS gauge inside
# 13TFJ, so each has a flow record to read alongside it (notebook 07). To run all four,
# wrap sections 2-12 in `for WINDOW_IDX in range(len(EXAMPLE_WINDOWS)):`.
EXAMPLE_WINDOWS = [
    {"site_no": "06401500", "name": "Angostura",   "lon": -103.4340, "lat": 43.3470},
    {"site_no": "06402600", "name": "Buffalo Gap", "lon": -103.2350, "lat": 43.4230},
    {"site_no": "06403700", "name": "Red Shirt",   "lon": -102.8921, "lat": 43.6724},
    {"site_no": "06408650", "name": "Scenic",      "lon": -102.5500, "lat": 43.7800},
]
# VERIFY: lon/lat for all but Red Shirt are approximate -- replace from the
# usgs_gauges layer of cheyenne_corridor_aoi.gpkg on first run.
WINDOW_IDX = 2                       # 2 = Red Shirt, the validated window
WINDOW     = EXAMPLE_WINDOWS[WINDOW_IDX]
CENTRE_LON, CENTRE_LAT = WINDOW["lon"], WINDOW["lat"]
WINDOW_HALF_M = 1000                 # half-width -> a 2 x 2 km window

# ---- Imagery ----
NAIP_YEAR = 2022                     # None -> newest epoch available (PC catalog ends 2023)

# ---- Structure features ----
# At 60 cm a tree is a rough, shadow-casting object and grass is a smooth one. These two
# numbers are what turn that sentence into a feature.
TEXTURE_WIN_M = 9.0                  # window for local roughness of NIR — crown-to-gap scale
SHADOW_FRAC   = 0.50                 # shadow = darker than this fraction of scene median brightness
SHADOW_REACH_M = 5.0                 # a segment is "shadow-adjacent" if shadow is within this far

# ---- Segmentation and clustering ----
SEGMENT_SIZE_M = 5.0                 # target superpixel width — a little under a crown
COMPACTNESS    = 0.15                # SLIC: low = boundaries follow colour, high = follow shape
N_CLUSTERS     = 10                  # k-means groups; enough to split sunlit from shaded canopy
SEED           = 42                  # fixed and recorded in the manifest

# ---- Rules that name each cluster (applied to cluster means, not pixels) ----
WATER_NDVI        = -0.30            # below this is open water
VEG_NDVI          =  0.15            # above this is photosynthesising
SHADOW_ADJ_WOODY  =  0.25            # green AND this shadow-adjacent -> standing woody canopy
SHADOW_ADJ_SHADE  =  0.75            # almost entirely shadow -> the shadow itself, not the crown
BARE_BRIGHT       =  0.55            # bright and not green -> sand, road, bare rock

# ---- Patches and validation ----
MIN_PATCH_M2   = 50.0                # drop woody blobs smaller than this (~ one small crown)
CLOSE_GAP_M    = 2.5                 # close gaps up to this wide before patching (crown shadow)
VALIDATION_N   = 300                 # stratified reference points for the group to interpret

# ---- Outputs ----
RUN_NAME = f"labels_{WINDOW['site_no']}_NAIP{NAIP_YEAR}"
FIG_SUBDIR = RUN_NAME               # figures/<run>/
OUT_DIR  = DATA_DIR / RUN_NAME
OUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Valley bottom : {VBET_RUN}")
print(f"Window        : {2 * WINDOW_HALF_M / 1000:.1f} x {2 * WINDOW_HALF_M / 1000:.1f} km "
      f"at {CENTRE_LAT:.4f}, {CENTRE_LON:.4f}")
print(f"Outputs       : {OUT_DIR}")

## 2. The valley bottom and the window

The valley bottom is the reason this is tractable. It is the riparian corridor delineated in `01s`
/ `01` from height above the stream and slope, and everything below happens inside it.

In [ ]:
VBET_GPKG   = VBET_DIR / f"{VBET_RUN}_valley_bottom.gpkg"
VBET_MASK   = VBET_DIR / f"{VBET_RUN}_valley_mask_{VBET_RES_M}m.tif"
VBET_HAND   = VBET_DIR / f"{VBET_RUN}_hand_{VBET_RES_M}m.tif"

missing = [p.name for p in (VBET_GPKG, VBET_MASK, VBET_HAND) if not p.exists()]
if missing:
    raise FileNotFoundError(
        f"Missing VBET outputs in {VBET_DIR}: {missing}\n"
        f"Run checks/01s_VBET_SmokeTest.ipynb (or 01_VBET_ValleyBottom.ipynb) first.")

valley    = gpd.read_file(VBET_GPKG, layer="valley_bottom")
flowlines = gpd.read_file(VBET_GPKG, layer="flowlines_classed")

# How much of the processed extent is valley bottom? This is the number that sets the
# real scaling problem (plan §13) — everything downstream only ever touches this fraction.
with rasterio.open(VBET_MASK) as src:
    VBET_EXTENT_KM2 = src.width * src.height * abs(src.transform.a * src.transform.e) / 1e6
VBET_VALLEY_KM2  = valley.area.sum() / 1e6
VBET_VALLEY_FRAC = VBET_VALLEY_KM2 / VBET_EXTENT_KM2

print(f"Valley bottom : {len(valley)} patches, {VBET_VALLEY_KM2:.1f} km2 ({valley.crs})")
print(f"Flowlines     : {len(flowlines)} reaches, "
      f"{flowlines.geometry.length.sum() / 1000:,.0f} km")
print(f"Coverage      : {100 * VBET_VALLEY_FRAC:.1f}% of the {VBET_EXTENT_KM2:,.0f} km2 "
      f"processed extent")

### Where the window sits

NAIP is delivered in NAD83 / UTM 13N (**EPSG:26913**) while the VBET pipeline works in WGS84 /
UTM 13N (**EPSG:32613**). Same zone, different datum, about 1–2 m apart — enough to matter at 60 cm.
We take the CRS from the imagery and bring everything else to it, rather than hardcoding either one.

In [ ]:
catalog = pystac_client.Client.open(
    "https://planetarycomputer.microsoft.com/api/stac/v1",
    modifier=planetary_computer.sign_inplace,
)

centre_wgs = gpd.GeoSeries.from_xy([CENTRE_LON], [CENTRE_LAT], crs=4326)
all_items  = list(catalog.search(collections=["naip"],
                                 intersects=centre_wgs.iloc[0].__geo_interface__).item_collection())

epochs = pd.DataFrame([{"year": int(i.properties["naip:year"]),
                        "gsd_m": i.properties["gsd"], "id": i.id} for i in all_items]
                      ).sort_values("year", ignore_index=True)
print("NAIP epochs over this point — the series Phase 4 uses for change:")
print(epochs.to_string(index=False))

# The imagery CRS decides the working grid, so read it off a real asset.
with rasterio.Env(GDAL_DISABLE_READDIR_ON_OPEN="EMPTY_DIR"):
    with rasterio.open(all_items[0].assets["image"].href) as src:
        NAIP_CRS = src.crs
print(f"\nWorking CRS: {NAIP_CRS} (VBET is {valley.crs})")

In [ ]:
# Build the window in the imagery CRS so WINDOW_HALF_M is genuinely metres.
c = centre_wgs.to_crs(NAIP_CRS).iloc[0]
window_poly = box(c.x - WINDOW_HALF_M, c.y - WINDOW_HALF_M,
                  c.x + WINDOW_HALF_M, c.y + WINDOW_HALF_M)
window_gdf  = gpd.GeoDataFrame(geometry=[window_poly], crs=NAIP_CRS)
window_wgs  = window_gdf.to_crs(4326).geometry.iloc[0]

fig, (a1, a2) = plt.subplots(1, 2, figsize=(7, 4), constrained_layout=True)

valley.plot(ax=a1, facecolor="#A5D6A7", edgecolor="#2E7D32", linewidth=0.5)
flowlines.plot(ax=a1, color="#1565C0", linewidth=0.5)
window_gdf.to_crs(valley.crs).boundary.plot(ax=a1, color="#C62828", linewidth=2)
a1.set_title(f"Window location (red)")

v_win = gpd.clip(valley.to_crs(NAIP_CRS), window_poly)
v_win.plot(ax=a2, facecolor="#A5D6A7", edgecolor="#2E7D32", linewidth=0.6)
gpd.clip(flowlines.to_crs(NAIP_CRS), window_poly).plot(ax=a2, color="#1565C0", linewidth=1.5)
window_gdf.boundary.plot(ax=a2, color="#C62828", linewidth=1.5)
a2.set_title(f"Window detail — valley bottom {v_win.area.sum() / 1e4:.0f} ha")

for ax in (a1, a2):
    ax.set_aspect("equal"); ax.set_xticks([]); ax.set_yticks([])
savefig("window_location")
plt.show()

## 3. NAIP imagery

`sign_inplace` fetches an anonymous SAS token and rewrites each asset URL with it — no account and
no requester-pays egress. **Tokens expire in about 45 minutes**: if a read below suddenly returns
HTTP 403, re-run this cell rather than debugging the raster.

In [ ]:
CHIP_PATH = OUT_DIR / f"naip_{RUN_NAME}.tif"

sel = [i for i in catalog.search(collections=["naip"],
                                 intersects=window_wgs.__geo_interface__).item_collection()
       if NAIP_YEAR is None or int(i.properties["naip:year"]) == NAIP_YEAR]
if not sel:
    raise ValueError(f"No NAIP for {NAIP_YEAR} over this window. Have: {list(epochs.year)}")

YEAR      = int(sel[0].properties["naip:year"])
NAIP_GSD  = float(sel[0].properties["gsd"])
NAIP_ITEM_IDS = sorted(i.id for i in sel)
print(f"{len(sel)} NAIP item(s) for {YEAR} at {NAIP_GSD} m: {NAIP_ITEM_IDS}")

t0 = time.time()
CHIP_CACHED = CHIP_PATH.exists()
if CHIP_CACHED:
    with rasterio.open(CHIP_PATH) as src:
        naip, naip_transform = src.read(), src.transform
    print(f"Chip already on disk: {CHIP_PATH.name}")
else:
    # merge() mosaics across quarter-quads and reads only the requested bounds.
    with rasterio.Env(GDAL_DISABLE_READDIR_ON_OPEN="EMPTY_DIR"):
        srcs = [rasterio.open(i.assets["image"].href) for i in sel]
        try:
            naip, naip_transform = merge(srcs, bounds=window_poly.bounds,
                                         res=NAIP_GSD, nodata=0)
        finally:
            for s in srcs:
                s.close()
    prof = dict(driver="GTiff", height=naip.shape[1], width=naip.shape[2], count=naip.shape[0],
                dtype=naip.dtype, crs=NAIP_CRS, transform=naip_transform,
                tiled=True, compress="LZW")
    with rasterio.open(CHIP_PATH, "w", **prof) as dst:
        dst.write(naip)
    print(f"Wrote {CHIP_PATH.name}")

READ_SECONDS = time.time() - t0
H, W = naip.shape[1], naip.shape[2]
# plotting_extent() reads shape[0] as the row count, so hand it one band — passing the
# full (4, H, W) stack makes it treat the 4 bands as 4 rows and returns a 2 m tall extent.
EXTENT = rasterio.plot.plotting_extent(naip[0], naip_transform)
print(f"Chip: {naip.shape} {naip.dtype} — {H * W / 1e6:.1f} M pixels in {READ_SECONDS:.0f} s")

### The cottonwood signal

NAIP band order is **R, G, B, NIR**. The colour-infrared view (NIR as red) is the one to read:
healthy broadleaf canopy reflects NIR strongly, so galleries light up against a cured prairie
matrix that has very little left to reflect by mid-July.

In [ ]:
f = naip.astype("float32") / 255.0
red, green, blue, nir = f[0], f[1], f[2], f[3]
brightness = f.mean(axis=0)

den  = nir + red
ndvi = np.where(den == 0, 0, (nir - red) / np.where(den == 0, 1, den)).astype("float32")

fig, ax = plt.subplots(1, 3, figsize=(8, 4.2), constrained_layout=True)
ax[0].imshow(np.moveaxis(naip[:3], 0, -1), extent=EXTENT)
ax[0].set_title(f"NAIP RGB — {YEAR}", fontsize=10)
ax[1].imshow(np.moveaxis(naip[[3, 0, 1]], 0, -1), extent=EXTENT)
ax[1].set_title("False color composite (NIR, R, G)", fontsize=10)
im = ax[2].imshow(ndvi, cmap="RdYlGn", vmin=-0.3, vmax=0.6, extent=EXTENT)
ax[2].set_title("NDVI", fontsize=10)
fig.colorbar(im, ax=ax[2], shrink=0.52)
for a in ax:
    a.set_xticks([]); a.set_yticks([])
savefig("naip_rgb_ndvi")
plt.show()

print(f"NDVI percentiles  1% {np.percentile(ndvi, 1):+.2f} | 50% {np.percentile(ndvi, 50):+.2f} "
      f"| 99% {np.percentile(ndvi, 99):+.2f}")
print(f"Pixels with NDVI > {VEG_NDVI}: {100 * (ndvi > VEG_NDVI).mean():.1f}% of the window — "
      f"green is scarce here, which is exactly why the gallery stands out.")

## 4. Terrain on the imagery grid

Three terrain layers come across from the VBET run and get resampled onto the 60 cm grid:

- the **valley mask** — where we are allowed to look;
- **HAND** — height above the nearest drainage, which is a floodplain-position variable;
- **distance to channel**, computed here from the NHD reaches.

HAND and distance are *context*, used to describe patches and to sanity-check results. They are
deliberately **not** fed to the clustering. The segmentation is already clipped to the valley
bottom, so "is in the valley bottom" carries no information where it is applied — using it as a
class rule would be circular.

In [ ]:
def to_naip_grid(path, resampling, dtype="float32"):
    """Reproject a VBET raster onto the NAIP grid."""
    dst = np.zeros((H, W), dtype=dtype)
    with rasterio.open(path) as src:
        reproject(rasterio.band(src, 1), dst,
                  dst_transform=naip_transform, dst_crs=NAIP_CRS,
                  src_nodata=src.nodata,
                  dst_nodata=(np.nan if dtype == "float32" else 0),
                  resampling=resampling)
    return dst

hand        = to_naip_grid(VBET_HAND, Resampling.bilinear)
valley_mask = to_naip_grid(VBET_MASK, Resampling.nearest, dtype="uint8") == 1

# Distance to the nearest NHD channel, in metres.
channel = rasterize(((g, 1) for g in flowlines.to_crs(NAIP_CRS).geometry),
                    out_shape=(H, W), transform=naip_transform,
                    fill=0, dtype="uint8", all_touched=True)
dist_channel = distance_transform_edt(channel == 0, sampling=NAIP_GSD).astype("float32")

VALLEY_HA = valley_mask.sum() * NAIP_GSD ** 2 / 1e4
print(f"Valley bottom in this window: {VALLEY_HA:,.0f} ha "
      f"({100 * valley_mask.mean():.0f}% of it)")

fig, ax = plt.subplots(1, 3, figsize=(8, 4.8), constrained_layout=True)
ax[0].imshow(valley_mask, cmap="Greens", extent=EXTENT)
ax[0].set_title("Valley mask (from VBET, 30 m)", fontsize=10)
im = ax[1].imshow(hand, cmap="viridis_r", vmax=25, extent=EXTENT)
ax[1].set_title("HAND (m)", fontsize=10); fig.colorbar(im, ax=ax[1], shrink=0.42)
im = ax[2].imshow(dist_channel, cmap="cividis_r", vmax=600, extent=EXTENT)
ax[2].set_title("Distance to NHD channel (m)", fontsize=10); fig.colorbar(im, ax=ax[2], shrink=0.42)
for a in ax:
    a.set_xticks([]); a.set_yticks([])
savefig("terrain_on_naip_grid")
plt.show()

## 5. Structure — what 60 cm buys us that 30 m cannot

Greenness alone will not separate a cottonwood from a wet meadow: in July both are green. What
separates them is that **a tree is a three-dimensional object**. At 60 cm that shows up two ways:

- **Roughness** — a crown is a mosaic of sunlit leaves and gaps, so local variance in NIR is high.
  Grass is smooth.
- **Shadow** — a tree casts one and grass does not. We find shadow as an absolute dark threshold,
  then ask of every segment: *is there shadow within a few metres of you?*

Shadow adjacency ends up doing most of the work, and it is the closest thing to a canopy height
measurement available without lidar.

In [ ]:
# Local standard deviation of NIR: sqrt(E[x^2] - E[x]^2) via two box filters — fast at 11 M px.
tw   = int(round(TEXTURE_WIN_M / NAIP_GSD)) | 1        # force odd
m1   = uniform_filter(nir, size=tw)
m2   = uniform_filter(nir * nir, size=tw)
texture = np.sqrt(np.clip(m2 - m1 * m1, 0, None)).astype("float32")

# Shadow as an absolute cut relative to the scene, not a fixed percentile: a percentile would
# declare a fixed share of any image to be shadow whether or not any shadow is present.
SHADOW_CUT  = SHADOW_FRAC * float(np.median(brightness))
shadow      = brightness < SHADOW_CUT
shadow_near = maximum_filter(shadow, size=int(round(SHADOW_REACH_M / NAIP_GSD))).astype("float32")

print(f"Shadow threshold : brightness < {SHADOW_CUT:.3f} "
      f"({100 * shadow.mean():.1f}% of pixels are shadow)")
print(f"Within {SHADOW_REACH_M:.0f} m of shadow: {100 * shadow_near.mean():.1f}% of pixels")

z = (slice(int(0.26 * H), int(0.63 * H)), slice(int(0.18 * W), int(0.57 * W)))  # a reach close-up
fig, ax = plt.subplots(1, 3, figsize=(8, 4.4), constrained_layout=True)
ax[0].imshow(np.moveaxis(naip[:3], 0, -1)[z])
ax[0].set_title("RGB", fontsize=9)
im = ax[1].imshow(texture[z], cmap="magma", vmax=np.percentile(texture, 99.5))
ax[1].set_title(f"Roughness — local SD of NIR ({TEXTURE_WIN_M:.0f} m)", fontsize=9)
fig.colorbar(im, ax=ax[1], shrink=0.52)
ax[2].imshow(np.moveaxis(naip[:3], 0, -1)[z])
ax[2].imshow(np.ma.masked_where(~shadow[z], shadow[z]),
             cmap=ListedColormap(["#0D47A1"]), alpha=0.85, interpolation="nearest")
ax[2].set_title("Detected shadow (blue)", fontsize=9)
for a in ax:
    a.set_xticks([]); a.set_yticks([])
savefig("structure_texture_shadow")
plt.show()

## 6. Segment

SLIC cuts the image into compact superpixels that follow colour edges. At a target width a little
under a crown, a tree becomes a handful of segments rather than being averaged into its background.

Segments are the unit of everything downstream: they average away NAIP's pixel noise, they cut the
problem from ten million pixels to a couple of hundred thousand objects, and — unlike pixels — they
have shape.

SLIC runs over the **whole window** and segments are filtered to the valley bottom afterwards.
Running it with a mask is far slower and gives the same answer.

In [ ]:
stack = np.stack([red, green, blue, nir, ndvi], axis=-1)
n_target = int(H * W / (SEGMENT_SIZE_M / NAIP_GSD) ** 2)

t0 = time.time()
segments = slic(stack, n_segments=n_target, compactness=COMPACTNESS, sigma=1.0,
                channel_axis=-1, start_label=1, enforce_connectivity=True)
SLIC_SECONDS = time.time() - t0
N_SEG = int(segments.max())
print(f"{N_SEG:,} segments in {SLIC_SECONDS:.0f} s "
      f"(target {n_target:,} at {SEGMENT_SIZE_M:.0f} m)")

zz = (slice(int(0.45 * H), int(0.52 * H)), slice(int(0.30 * W), int(0.40 * W)))
fig, ax = plt.subplots(1, 2, figsize=(8, 3.2), constrained_layout=True)
ax[0].imshow(np.moveaxis(naip[:3], 0, -1)[zz])
ax[0].set_title("True Color (RGB)")
ax[1].imshow(mark_boundaries(np.moveaxis(naip[:3], 0, -1)[zz].astype("float32") / 255.0,
                             segments[zz], color=(1, 1, 0), mode="subpixel"))
ax[1].set_title("SLIC segments")
for a in ax:
    a.set_xticks([]); a.set_yticks([])
savefig("segmentation_closeup")
plt.show()

## 7. Describe each segment

Per-segment means come from `np.bincount` with weights — one pass over the array per feature,
which is what makes this fast enough to be interactive at 11 M pixels.

In [ ]:
lab   = segments.ravel()
nlab  = N_SEG + 1
count = np.bincount(lab, minlength=nlab).astype("float64")
segmean = lambda a: np.bincount(lab, weights=a.ravel(), minlength=nlab) / np.maximum(count, 1)

t0 = time.time()
feat = pd.DataFrame({
    "n_px":         count,
    # spectral
    "red":          segmean(red),
    "green":        segmean(green),
    "blue":         segmean(blue),
    "nir":          segmean(nir),
    "brightness":   segmean(brightness),
    "ndvi":         segmean(ndvi),
    "ndvi_sd":      np.sqrt(np.clip(segmean(ndvi * ndvi) - segmean(ndvi) ** 2, 0, None)),
    # structure
    "texture":      segmean(texture),
    "shadow_adj":   segmean(shadow_near),
    # context — described, never clustered on
    "hand":         segmean(np.nan_to_num(hand, nan=0.0)),
    "dist_channel": segmean(dist_channel),
    "valley_frac":  segmean(valley_mask.astype("float32")),
})
feat = feat[feat.n_px > 0]
feat["area_m2"] = feat.n_px * NAIP_GSD ** 2
print(f"Features for {len(feat):,} segments in {time.time() - t0:.1f} s")

# Keep only segments that are mostly inside the valley bottom.
inside = feat[feat.valley_frac >= 0.5].copy()
print(f"Inside the valley bottom: {len(inside):,} segments "
      f"({inside.area_m2.sum() / 1e4:,.0f} ha, median segment {inside.area_m2.median():.0f} m2)")

## 8. Cluster

k-means on six standardised features — colour, greenness, roughness, shadow adjacency. No class
labels go in. What comes out are the groups the imagery actually contains; naming them is the next
step and a separate one.

The seed is fixed and recorded in the manifest, so this is reproducible rather than merely
repeatable.

In [ ]:
CLUSTER_FEATURES = ["ndvi", "ndvi_sd", "texture", "shadow_adj", "brightness", "nir"]

X  = StandardScaler().fit_transform(inside[CLUSTER_FEATURES].values)
km = KMeans(n_clusters=N_CLUSTERS, random_state=SEED, n_init=10).fit(X)
inside["cluster"] = km.labels_

profile = (inside.groupby("cluster")
           .agg(n_segments=("n_px", "size"),
                area_ha=("area_m2", lambda s: s.sum() / 1e4),
                **{c: (c, "mean") for c in CLUSTER_FEATURES + ["hand", "dist_channel"]})
           .sort_values("ndvi", ascending=False).round(3))

print("Cluster profile — this table is the thing to read:\n")
print(profile.to_string())

### Reading the profile

Look down the `ndvi` and `shadow_adj` columns together. Three groups should stand out:

- strongly negative NDVI with very low NIR, sitting metres from the channel — **open water**;
- green, rough, and **shadow-adjacent**, close to the channel and low above it — **woody canopy**;
- green, smooth, and **not** shadow-adjacent — **herbaceous**, the wet meadow on the point bars.

Everything else is some flavour of bare or sparsely vegetated prairie. If that structure is not
visible in your run, change `N_CLUSTERS` rather than bending the thresholds below.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(8.5, 4), constrained_layout=True)

# The woody cluster is small in area and large in importance, so scale marker size by the
# square root of area with a floor — linear scaling makes exactly the interesting ones vanish.
size = 80 + 1100 * np.sqrt(profile.area_ha / profile.area_ha.max())

ax[0].axhspan(SHADOW_ADJ_WOODY, SHADOW_ADJ_SHADE, xmin=0, xmax=1, color="#1B5E20", alpha=0.06)
sc = ax[0].scatter(profile.ndvi, profile.shadow_adj, s=size,
                   c=profile.texture, cmap="magma", edgecolor="k", linewidth=0.7, zorder=3)
for cid, r in profile.iterrows():
    ax[0].annotate(int(cid), (r.ndvi, r.shadow_adj), fontsize=8, zorder=4,
                   xytext=(12, 2), textcoords="offset points", weight="bold")
ax[0].axvline(VEG_NDVI, color="#2E7D32", ls="--", lw=1.2)
ax[0].axvline(WATER_NDVI, color="#1565C0", ls="--", lw=1.2)
ax[0].axhline(SHADOW_ADJ_WOODY, color="#6D4C41", ls="--", lw=1.2)
ax[0].axhline(SHADOW_ADJ_SHADE, color="#6D4C41", ls=":", lw=1.2)
ax[0].annotate("green AND shadow-adjacent\n= woody canopy", (0.97, 0.68), xycoords="axes fraction",
               ha="right", fontsize=8, color="#1B5E20", weight="bold")
ax[0].set_xlabel("mean NDVI"); ax[0].set_ylabel("shadow adjacency")
ax[0].set_title("Clusters (size = area, colour = roughness)",
                fontsize=10)
fig.colorbar(sc, ax=ax[0], shrink=0.8, label="texture")

ax[1].hist(inside.ndvi, bins=120, color="#90A4AE")
ax[1].axvline(VEG_NDVI, color="#2E7D32", ls="--", lw=1.5, label=f"veg > {VEG_NDVI}")
ax[1].axvline(WATER_NDVI, color="#1565C0", ls="--", lw=1.5, label=f"water < {WATER_NDVI}")
ax[1].set_xlabel("segment mean NDVI"); ax[1].set_ylabel("segments")
ax[1].set_title("NDVI Thresholds",
                fontsize=10)
ax[1].legend()
savefig("cluster_profile")
plt.show()

## 9. Name the clusters

The rules are applied **to cluster means, not to pixels**. That is the whole design: there are
`N_CLUSTERS` decisions here, every one of them is printed below, and every one is physical enough
to argue with. First match wins.

| Class | Rule |
|---|---|
| `water` | NDVI below `WATER_NDVI` |
| `shadow` | almost entirely shadow-adjacent — the cast shadow, not the crown |
| `riparian_woody` | green **and** shadow-adjacent |
| `herbaceous` | green, but open |
| `bare_bright` | not green, bright — sand, road, bare rock |
| `sparse_bare` | everything else |

In [ ]:
CLASS_ORDER  = ["water", "riparian_woody", "shadow", "herbaceous", "sparse_bare", "bare_bright"]
CLASS_CODE   = {c: i + 1 for i, c in enumerate(CLASS_ORDER)}
CLASS_COLOUR = {"water": "#1565C0", "riparian_woody": "#1B5E20", "shadow": "#6D4C41",
                "herbaceous": "#9CCC65", "sparse_bare": "#D7CCC8", "bare_bright": "#FFF8E1"}

def assign_class(r):
    """Name one cluster from its mean features. First match wins."""
    if r.ndvi < WATER_NDVI:                                        return "water"
    if r.shadow_adj >= SHADOW_ADJ_SHADE:                           return "shadow"
    if r.ndvi >= VEG_NDVI and r.shadow_adj >= SHADOW_ADJ_WOODY:    return "riparian_woody"
    if r.ndvi >= VEG_NDVI:                                         return "herbaceous"
    if r.brightness >= BARE_BRIGHT:                                return "bare_bright"
    return "sparse_bare"

profile["class"] = profile.apply(assign_class, axis=1)
inside["class"]  = inside.cluster.map(profile["class"])

print("Every decision this notebook makes:\n")
print(profile[["n_segments", "area_ha", "ndvi", "texture", "shadow_adj",
               "brightness", "hand", "dist_channel", "class"]].to_string())

areas = (inside.groupby("class").area_m2.sum() / 1e4).reindex(CLASS_ORDER).fillna(0)
print("\nArea by class (ha, inside the valley bottom):")
print(areas.round(1).to_string())

In [ ]:
# Paint the classes back onto the grid.
lut = np.zeros(nlab, dtype="uint8")
lut[inside.index.values] = inside["class"].map(CLASS_CODE).values
class_map = lut[segments]

cmap = ListedColormap([CLASS_COLOUR[c] for c in CLASS_ORDER])
shown = np.ma.masked_where(class_map == 0, class_map)

fig, ax = plt.subplots(1, 2, figsize=(8.5, 4.6), constrained_layout=True)
ax[0].imshow(np.moveaxis(naip[:3], 0, -1), extent=EXTENT)
ax[0].set_title(f"True Color (RGB) — {YEAR}")
ax[1].imshow(shown, cmap=cmap, vmin=1, vmax=len(CLASS_ORDER),
             extent=EXTENT, interpolation="nearest")
ax[1].set_title("Labeled classes")
for a in ax:
    a.set_xticks([]); a.set_yticks([])

handles = [plt.Rectangle((0, 0), 1, 1, fc=CLASS_COLOUR[c], ec="k", lw=0.4) for c in CLASS_ORDER]
ax[1].legend(handles, CLASS_ORDER, loc="lower left", fontsize=9, framealpha=0.92)
savefig("class_map")
plt.show()

## 10. Gallery patches

The class raster is a per-segment result. Patches are the analysis unit Phase 4 needs: stable
polygons that Landsat index series get extracted *within*, so condition can be trended inside a
fixed footprint rather than compared between two imperfect maps.

A small morphological closing first: a crown and the shadow it casts are one tree, and leaving the
shadow as a hole would fragment every patch.

In [ ]:
woody = class_map == CLASS_CODE["riparian_woody"]
k = int(round(CLOSE_GAP_M / NAIP_GSD)) | 1
closed = binary_closing(woody, structure=np.ones((k, k), bool))

labels_cc, n_cc = cc_label(closed)
sizes = np.bincount(labels_cc.ravel()); sizes[0] = 0
keep  = np.flatnonzero(sizes >= MIN_PATCH_M2 / NAIP_GSD ** 2)
gallery = np.isin(labels_cc, keep)

polys = [shape(g) for g, v in shapes(gallery.astype("uint8"), mask=gallery,
                                     transform=naip_transform) if v == 1]
patches = gpd.GeoDataFrame(geometry=polys, crs=NAIP_CRS)
patches = patches[patches.area >= MIN_PATCH_M2].reset_index(drop=True)
patches.insert(0, "patch_id", patches.index + 1)
patches["class_name"] = "riparian_woody"
patches["area_m2"]    = patches.area.round(1)

# Elongation — the ratio of a patch's long axis to its short one, from the principal axes of
# its outline. A gallery is a ribbon along the channel; a shelterbelt or a yard tree is not.
def elongation(g):
    xy = np.asarray(g.exterior.coords)[:-1]
    if len(xy) < 3:
        return np.nan
    ev = np.linalg.eigvalsh(np.cov((xy - xy.mean(axis=0)).T))
    return round(float(np.sqrt(ev[1] / ev[0])), 2) if ev[0] > 1e-12 else np.nan

patches["elongation"] = patches.geometry.map(elongation)

centres  = patches.geometry.representative_point()
row, col = rasterio.transform.rowcol(naip_transform, centres.x.values, centres.y.values)
row = np.clip(row, 0, H - 1); col = np.clip(col, 0, W - 1)
patches["hand_m"]      = np.round(np.nan_to_num(hand)[row, col], 2)
patches["dist_chan_m"] = np.round(dist_channel[row, col], 1)

GALLERY_HA = patches.area.sum() / 1e4
print(f"Components {n_cc:,} -> {len(patches):,} patches kept (>= {MIN_PATCH_M2:.0f} m2)")
print(f"Gallery area : {GALLERY_HA:,.1f} ha "
      f"({100 * GALLERY_HA / VALLEY_HA:.1f}% of the valley bottom in this window)")
print(f"Patch size   : median {patches.area_m2.median():,.0f} m2, "
      f"largest {patches.area_m2.max():,.0f} m2")

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(9.5, 4.4), constrained_layout=True,
                       gridspec_kw={"width_ratios": [0.8, 1.45]})
for a, sl, ttl in [(ax[0], (slice(None), slice(None)), "Whole window"),
                   (ax[1], z, "Close up")]:
    a.imshow(np.moveaxis(naip[:3], 0, -1)[sl])
    a.imshow(np.ma.masked_where(~gallery[sl], gallery[sl]),
             cmap=ListedColormap(["#00E676"]), alpha=0.55, interpolation="nearest")
    # a.set_title(f"{ttl}")
    a.set_xticks([]); a.set_yticks([])
savefig("gallery_patches")
plt.show()

### Look at this before trusting it

Two failure modes are visible in almost every run and both are worth naming out loud:

- **Planted trees are not gallery forest.** Shelterbelts and yard trees around ranch buildings are
  woody, shadow-casting and green, so they classify exactly like cottonwood. `elongation` and
  `dist_chan_m` are on every patch so they can be screened — but the screen is a decision for the
  group, not a default applied here.
- **Willow and shrub thicket classify as woody too.** That is the known limit of 4-band NAIP
  (§5.1), and it is why the class is called `riparian_woody`.

## 11. A validation sample for the group

Unsupervised labels do not exempt the work from validation, and this is the one irreducible piece
of manual interpretation. It is also the step where local knowledge of the river beats a remote
analyst, so it is designed as a **group activity** rather than a solo chore.

The sample below is **stratified random by mapped class** and is kept strictly separate from the
labels above — it is never used for training. Open the GeoPackage in QGIS over the NAIP chip, and
fill in `label` (what it really is), `confidence`, and `interpreter`.

In [ ]:
rng  = np.random.default_rng(SEED)
rows_, cols_ = np.nonzero(class_map > 0)
codes = class_map[rows_, cols_]

per_class = max(1, VALIDATION_N // len(CLASS_ORDER))
pick = []
for cname in CLASS_ORDER:
    idx = np.nonzero(codes == CLASS_CODE[cname])[0]
    if idx.size == 0:
        print(f"  {cname:15s}: not present in this window")
        continue
    take = rng.choice(idx, size=min(per_class, idx.size), replace=False)
    pick.append(take)
    print(f"  {cname:15s}: {len(take):3d} points")
pick = np.concatenate(pick)

xs, ys = rasterio.transform.xy(naip_transform, rows_[pick], cols_[pick])
val = gpd.GeoDataFrame(
    {"point_id":    np.arange(1, len(pick) + 1),
     "pred_class":  [CLASS_ORDER[c - 1] for c in codes[pick]],
     "label":       "",          # <- the interpreter fills these three in
     "confidence":  "",
     "interpreter": "",
     "notes":       ""},
    geometry=gpd.points_from_xy(xs, ys), crs=NAIP_CRS)

print(f"\n{len(val)} reference points, stratified by mapped class, seed {SEED}")

## 12. Save, measure, record

In [ ]:
# ---- Class raster ----
CLASS_TIF = OUT_DIR / f"{RUN_NAME}_classes.tif"
prof = dict(driver="GTiff", height=H, width=W, count=1, dtype="uint8",
            crs=NAIP_CRS, transform=naip_transform, nodata=0,
            tiled=True, compress="deflate")
with rasterio.open(CLASS_TIF, "w", **prof) as dst:
    dst.write(class_map, 1)
    dst.write_colormap(1, {CLASS_CODE[c]: tuple(int(CLASS_COLOUR[c][i:i + 2], 16)
                                                for i in (1, 3, 5)) + (255,)
                           for c in CLASS_ORDER})

# ---- Vectors ----
LABELS_GPKG = OUT_DIR / f"{RUN_NAME}.gpkg"
patches.to_file(LABELS_GPKG, layer="gallery_patches", driver="GPKG")
val.to_file(LABELS_GPKG, layer="validation_sample", driver="GPKG")
window_gdf.to_file(LABELS_GPKG, layer="window", driver="GPKG")

print(f"Classes    -> {CLASS_TIF.name}")
print(f"Patches    -> {LABELS_GPKG.name}::gallery_patches ({len(patches):,})")
print(f"Validation -> {LABELS_GPKG.name}::validation_sample ({len(val)})")

### Cost per tile

The Phase 2 gate asks for a measured cost, so the corridor-scale decision (local/CyVerse versus
Earth Engine) rests on numbers rather than intuition. This extrapolates the window to **13TFJ**,
counting only the valley-bottom fraction, since that is all we ever process.

In [ ]:
# Scale on valley-bottom area, not on tile area — and take the valley fraction from the VBET
# run, not from this window. The window was deliberately centred on the river, so 65% of it is
# valley bottom while only ~15% of the wider extent is. Using the window fraction would inflate
# the estimate by the ratio between them.
WINDOW_KM2        = (2 * WINDOW_HALF_M / 1000) ** 2
WINDOW_VALLEY_KM2 = VALLEY_HA / 100
TILE_VALLEY_KM2   = 100 * 100 * VBET_VALLEY_FRAC
scale = TILE_VALLEY_KM2 / WINDOW_VALLEY_KM2

cost = pd.DataFrame({
    "this window": [f"{WINDOW_VALLEY_KM2:,.1f}", f"{H * W / 1e6:,.1f}", f"{N_SEG:,}",
                    f"{READ_SECONDS / 60:.1f}", f"{SLIC_SECONDS / 60:.1f}"],
    "13TFJ (extrapolated)": [f"{TILE_VALLEY_KM2:,.0f}", f"{H * W / 1e6 * scale:,.0f}",
                             f"{int(N_SEG * scale):,}",
                             f"{READ_SECONDS * scale / 60:,.0f}",
                             f"{SLIC_SECONDS * scale / 60:,.0f}"],
}, index=["valley bottom km2", "M pixels", "segments",
          "imagery read (min)", "segmentation (min)"])
print(cost.to_string())
print(f"\nScale factor {scale:,.0f}x, from a valley bottom that is "
      f"{100 * VBET_VALLEY_FRAC:.0f}% of the {VBET_RUN} extent.")
if CHIP_CACHED:
    print("NOTE: the chip was read from disk, so 'imagery read' is not a download time. "
          "Delete\n      the chip and re-run for a true network figure.")
print("The read is network-bound and embarrassingly parallel; the segmentation is CPU-bound and\n"
      "runs window by window, so neither needs the whole tile in memory. Compare these numbers\n"
      "against the Earth Engine question in §13 of the plan — that decision wants evidence.")

In [ ]:
import subprocess

def git_commit():
    try:
        return subprocess.run(["git", "rev-parse", "--short", "HEAD"], cwd=REPO,
                              capture_output=True, text=True).stdout.strip() or None
    except Exception:
        return None

manifest = {
    "run_name":    RUN_NAME,
    "notebook":    "03_NAIP_Segmentation_Labels.ipynb",
    "created_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    "git_commit":  git_commit(),
    "inputs": {
        "vbet_run":        VBET_RUN,
        "vbet_manifest":   f"{VBET_RUN}.manifest.json",
        "naip_stac_items": NAIP_ITEM_IDS,     # the catalog is not immutable — pin the items
        "naip_year":       YEAR,
        "naip_gsd_m":      NAIP_GSD,
        "stac_endpoint":   "https://planetarycomputer.microsoft.com/api/stac/v1",
    },
    "parameters": {
        "centre_lon_lat":   [CENTRE_LON, CENTRE_LAT],
        "window_half_m":    WINDOW_HALF_M,
        "crs":              str(NAIP_CRS),
        "texture_win_m":    TEXTURE_WIN_M,
        "shadow_frac":      SHADOW_FRAC,
        "shadow_reach_m":   SHADOW_REACH_M,
        "segment_size_m":   SEGMENT_SIZE_M,
        "compactness":      COMPACTNESS,
        "n_clusters":       N_CLUSTERS,
        "seed":             SEED,
        "cluster_features": CLUSTER_FEATURES,
        "rules": {"water_ndvi": WATER_NDVI, "veg_ndvi": VEG_NDVI,
                  "shadow_adj_woody": SHADOW_ADJ_WOODY,
                  "shadow_adj_shade": SHADOW_ADJ_SHADE, "bare_bright": BARE_BRIGHT},
        "min_patch_m2":     MIN_PATCH_M2,
        "close_gap_m":      CLOSE_GAP_M,
        "validation_n":     VALIDATION_N,
    },
    "environment": {
        "python":       sys.version.split()[0],
        "geopandas":    gpd.__version__,
        "rasterio":     rasterio.__version__,
        "scikit-image": __import__("skimage").__version__,
        "scikit-learn": __import__("sklearn").__version__,
    },
    "results": {
        "window_km2":          round(WINDOW_KM2, 2),
        "valley_bottom_ha":    round(VALLEY_HA, 1),
        "vbet_valley_frac":    round(VBET_VALLEY_FRAC, 4),
        "tile_scale_factor":   round(scale, 1),
        "n_segments":          N_SEG,
        "n_segments_in_mask":  int(len(inside)),
        "area_ha_by_class":    {k: round(v, 2) for k, v in areas.items()},
        "gallery_ha":          round(GALLERY_HA, 2),
        "gallery_pct_valley":  round(100 * GALLERY_HA / VALLEY_HA, 2),
        "n_patches":           int(len(patches)),
        "n_validation_points": int(len(val)),
        "cluster_class_map":   profile["class"].to_dict(),
        "read_seconds":        round(READ_SECONDS, 1),
        "slic_seconds":        round(SLIC_SECONDS, 1),
    },
    "outputs": [CLASS_TIF.name, LABELS_GPKG.name, CHIP_PATH.name],
}

manifest_path = RUNS_DIR / f"{RUN_NAME}.manifest.json"
manifest_path.write_text(json.dumps(manifest, indent=2, default=str) + "\n")
print(f"Manifest -> {manifest_path.relative_to(REPO)}")
print(json.dumps(manifest["results"], indent=2, default=str))

## What comes next

- **Interpret the validation sample** as a group, then compute per-class precision / recall / F1
  with blocked cross-validation by reach. That number is the Phase 2 gate.
- **Run the other epochs.** Change `NAIP_YEAR` — Red Shirt has 2012, 2014, 2016, 2018, 2020, 2021
  and 2022 — and the patches become the fixed units for the change work in notebook 06.
- **Aggregate these labels to 30 m** with a purity filter, which is what trains the Landsat model
  in notebooks 04–05.
- **Check 3DEP lidar coverage.** A canopy height model is the one thing that would turn
  `riparian_woody` into `cottonwood` honestly.